# 2.2 Create Speech Dataset Splits

Purpose:
- Load the corrected speech audio manifest
- Create speaker-level train/validation/test splits
- Prevent speaker leakage
- Save split manifest

In [1]:
import pandas as pd

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    SPEECH_INTERIM_MANIFEST_PATH,
    SPEECH_SPLIT_MANIFEST_PATH,
)
from rural_stroke_assist.preprocessing.splitting import (
    add_group_split,
    summarize_split_balance,
)

In [2]:
speech_df = pd.read_csv(SPEECH_INTERIM_MANIFEST_PATH)

speech_df.head()

,path,file_name,extension,label,torgo_group,speaker_id,readable,duration_seconds,sample_rate,num_samples
0,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0001.wav,.wav,dysarthric,M_Dys,wav_arrayMic_M01S01,True,9.30,16000.0,148800.0
1,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0002.wav,.wav,dysarthric,M_Dys,wav_arrayMic_M01S01,True,7.80,16000.0,124800.0
2,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0003.wav,.wav,dysarthric,M_Dys,wav_arrayMic_M01S01,True,8.10,16000.0,129600.0
3,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0004.wav,.wav,dysarthric,M_Dys,wav_arrayMic_M01S01,True,6.15,16000.0,98400.0
4,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0005.wav,.wav,dysarthric,M_Dys,wav_arrayMic_M01S01,True,12.30,16000.0,196800.0


In [3]:
speech_df[["label", "torgo_group", "speaker_id", "readable"]].head()

,label,torgo_group,speaker_id,readable
0,dysarthric,M_Dys,wav_arrayMic_M01S01,True
1,dysarthric,M_Dys,wav_arrayMic_M01S01,True
2,dysarthric,M_Dys,wav_arrayMic_M01S01,True
3,dysarthric,M_Dys,wav_arrayMic_M01S01,True
4,dysarthric,M_Dys,wav_arrayMic_M01S01,True


In [4]:
speech_df["label"].value_counts()

label
control       11455
dysarthric     6178
Name: count, dtype: int64

In [5]:
speech_df["speaker_id"].value_counts().sort_index()

speaker_id
wav_arrayMic_F01         134
wav_arrayMic_F03S01      204
wav_arrayMic_F03S02      435
wav_arrayMic_F03S03      209
wav_arrayMic_F04S01      199
wav_arrayMic_F04S02      249
wav_arrayMic_FC01S01     256
wav_arrayMic_FC02S02     269
wav_arrayMic_FC02S03     992
wav_arrayMic_FC03S01     400
wav_arrayMic_FC03S02     386
wav_arrayMic_FC03S03     208
wav_arrayMic_M01S01      100
wav_arrayMic_M01S02      286
wav_arrayMic_M02S01      240
wav_arrayMic_M02S02      160
wav_arrayMic_M03S02      416
wav_arrayMic_M04S01      126
wav_arrayMic_M04S02      294
wav_arrayMic_M05S01      128
wav_arrayMic_MC01S01     329
wav_arrayMic_MC01S02     360
wav_arrayMic_MC01S03     419
wav_arrayMic_MC02S01     388
wav_arrayMic_MC02S02     307
wav_arrayMic_MC03S01     600
wav_arrayMic_MC03S02     300
wav_arrayMic_MC04S01     648
wav_arrayMic_MC04S02     373
wav_headMic_F01          134
wav_headMic_F03S01       204
wav_headMic_F03S02       159
wav_headMic_F03S03       214
wav_headMic_F04S02       250
wav

In [6]:
speech_model_df = speech_df[
    (speech_df["readable"] == True)
    & (speech_df["label"] != "unknown")
    & (speech_df["speaker_id"] != "unknown")
].copy()

speech_model_df.shape

(17631, 10)

In [7]:
speech_model_df.groupby(["speaker_id", "label"]).size()

speaker_id            label     
wav_arrayMic_F01      dysarthric     134
wav_arrayMic_F03S01   dysarthric     204
wav_arrayMic_F03S02   dysarthric     435
wav_arrayMic_F03S03   dysarthric     209
wav_arrayMic_F04S01   dysarthric     199
wav_arrayMic_F04S02   dysarthric     249
wav_arrayMic_FC01S01  control        256
wav_arrayMic_FC02S02  control        269
wav_arrayMic_FC02S03  control        992
wav_arrayMic_FC03S01  control        400
wav_arrayMic_FC03S02  control        386
wav_arrayMic_FC03S03  control        208
wav_arrayMic_M01S01   dysarthric     100
wav_arrayMic_M01S02   dysarthric     286
wav_arrayMic_M02S01   dysarthric     240
wav_arrayMic_M02S02   dysarthric     160
wav_arrayMic_M03S02   dysarthric     416
wav_arrayMic_M04S01   dysarthric     126
wav_arrayMic_M04S02   dysarthric     294
wav_arrayMic_M05S01   dysarthric     128
wav_arrayMic_MC01S01  control        329
wav_arrayMic_MC01S02  control        360
wav_arrayMic_MC01S03  control        419
wav_arrayMic_MC02S01  co

In [8]:
speech_split_df = add_group_split(
    df=speech_model_df,
    group_column="speaker_id",
    label_column="label",
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
    random_state=42,
)

speech_split_df.head()

,path,file_name,extension,label,torgo_group,speaker_id,readable,duration_seconds,sample_rate,num_samples,split
0,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0001.wav,.wav,dysarthric,M_Dys,wav_arrayMic_M01S01,True,9.30,16000.0,148800.0,train
1,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0002.wav,.wav,dysarthric,M_Dys,wav_arrayMic_M01S01,True,7.80,16000.0,124800.0,train
2,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0003.wav,.wav,dysarthric,M_Dys,wav_arrayMic_M01S01,True,8.10,16000.0,129600.0,train
3,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0004.wav,.wav,dysarthric,M_Dys,wav_arrayMic_M01S01,True,6.15,16000.0,98400.0,train
4,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0005.wav,.wav,dysarthric,M_Dys,wav_arrayMic_M01S01,True,12.30,16000.0,196800.0,train


In [9]:
speech_split_df.groupby(["split", "speaker_id"]).size()

split  speaker_id          
test   wav_arrayMic_F01         134
       wav_arrayMic_FC03S01     400
       wav_arrayMic_MC03S02     300
       wav_headMic_FC03S03      207
       wav_headMic_M01S02       286
       wav_headMic_M02S02       169
       wav_headMic_M05S02       393
       wav_headMic_MC01S02      360
       wav_headMic_MC03S01      600
train  wav_arrayMic_F03S01      204
       wav_arrayMic_F03S03      209
       wav_arrayMic_F04S01      199
       wav_arrayMic_FC01S01     256
       wav_arrayMic_FC03S02     386
       wav_arrayMic_FC03S03     208
       wav_arrayMic_M01S01      100
       wav_arrayMic_M01S02      286
       wav_arrayMic_M02S01      240
       wav_arrayMic_M02S02      160
       wav_arrayMic_M03S02      416
       wav_arrayMic_M04S01      126
       wav_arrayMic_M04S02      294
       wav_arrayMic_M05S01      128
       wav_arrayMic_MC01S01     329
       wav_arrayMic_MC01S02     360
       wav_arrayMic_MC01S03     419
       wav_arrayMic_MC02S01     388


In [10]:
split_summary = summarize_split_balance(
    df=speech_split_df,
    split_column="split",
    label_column="label",
)

split_summary

,split,label,count,percentage
0,test,control,1867,65.53
1,test,dysarthric,982,34.47
2,train,control,6679,61.28
3,train,dysarthric,4221,38.72
4,val,control,2909,74.94
5,val,dysarthric,973,25.06


In [11]:
speaker_splits = (
    speech_split_df[["speaker_id", "split"]]
    .drop_duplicates()
    .groupby("speaker_id")["split"]
    .nunique()
)

leaky_speakers = speaker_splits[speaker_splits > 1]

leaky_speakers

Series([], Name: split, dtype: int64)

In [12]:
assert len(leaky_speakers) == 0, "Speaker leakage detected!"

In [13]:
SPEECH_SPLIT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

speech_split_df.to_csv(SPEECH_SPLIT_MANIFEST_PATH, index=False)

print("Saved speech split manifest to:", SPEECH_SPLIT_MANIFEST_PATH)

Saved speech split manifest to: C:\Users\Asus\Downloads\Uni_work\Grad-Project\rural-stroke-assist\data\processed\speech_split_manifest.csv


## Split Decision

Speech data was split by `speaker_id`, not by individual audio file.

This prevents speaker leakage, where the same speaker's voice appears in both training and testing sets. Speaker leakage would make evaluation results artificially optimistic.

The split is therefore more realistic for evaluating whether a model generalizes to unseen speakers.